# Introductory task for Applied Artificial Intelligence Lab

## Relevant information

I am an ERASMUS student, so I am only here for this semester. 

During my last semester in France, I passed a class about data analysis where I learned how to process and clean the data. I also learned about PCA, CCA, FCA and clustering algorithms.

Another class was about regression algorithms, where I applied linear regression, logistic regression with regularization such as Lasso and Ridge.

# Rainy or Sunny classifier

In [9]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
import re
import csv
import matplotlib.pyplot as plt
import torch

In [3]:
weather_to_idx = {"rainy": 0, "sunny": 1}

class WeatherDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path)
        label_name = self.img_labels.iloc[idx, 1]
        label = weather_to_idx[label_name]
        if self.transform:
            image = self.transform(image)
        return image, label

In [4]:
img_path = "data/all_except_marked"

data = []
for filename in os.listdir(img_path):
    if filename.lower().endswith(('.png')):
        match = re.match(r"image_\d*(\D+)_(\w+)\.\w+", filename)
        if match:
            style = match.group(1)   # "realistic"
            weather = match.group(2) # "rainy"
            data.append([filename, weather, style])
        else:
            print(f"Nom inattendu : {filename}")
            
csv_path = os.path.join(img_path, "image_metadata.csv")
with open(csv_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["filename", "weather", "style"])
    writer.writerows(data)

In [5]:
first_dataset = WeatherDataset(annotations_file=csv_path, img_dir=img_path)

train_size = int(0.8 * len(first_dataset))
test_size = len(first_dataset) - train_size

train_dataset, test_dataset = random_split(first_dataset, [train_size, test_size])

In [6]:
len(train_dataset), len(test_dataset)

(128, 32)

In [8]:
image, label = train_dataset[0]
image, label

(tensor([[[166, 176, 180,  ..., 192, 195, 192],
          [163, 172, 182,  ..., 188, 193, 192],
          [155, 173, 181,  ..., 190, 191, 190],
          ...,
          [ 97,  94,  98,  ..., 180, 182, 184],
          [ 93,  96,  99,  ..., 185, 183, 184],
          [ 94, 100, 102,  ..., 184, 181, 187]],
 
         [[148, 158, 165,  ..., 197, 196, 192],
          [148, 155, 166,  ..., 193, 195, 192],
          [139, 157, 166,  ..., 192, 190, 188],
          ...,
          [ 91,  88,  91,  ..., 172, 174, 174],
          [ 90,  93,  94,  ..., 181, 180, 176],
          [ 90,  96,  96,  ..., 178, 176, 181]],
 
         [[124, 134, 144,  ..., 192, 192, 186],
          [120, 129, 142,  ..., 187, 189, 187],
          [115, 131, 141,  ..., 187, 185, 184],
          ...,
          [102,  99, 101,  ..., 170, 171, 173],
          [101, 103, 103,  ..., 176, 176, 176],
          [101, 106, 106,  ..., 176, 175, 178]]], dtype=torch.uint8),
 1)

In [11]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

first_dataset = WeatherDataset(annotations_file=csv_path, img_dir=img_path, transform=transform)

train_size = int(0.8 * len(first_dataset))
test_size = len(first_dataset) - train_size

train_dataset, test_dataset = random_split(first_dataset, [train_size, test_size])

In [12]:
image, label = train_dataset[0]
image.shape

TypeError: pic should be PIL Image or ndarray. Got <class 'torch.Tensor'>